# E2B Example: Parallel CI/CD Runner

This example uses one orchestrator and three isolated E2B sandboxes:

- `test_runner`: runs tests
- `lint_runner`: runs lint/format checks
- `preview_runner`: starts a tiny app and returns a preview URL

The final answer is a synthesized CI-style status report.

In [ ]:
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "src" / "agents").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "agents").exists():
            repo_root = candidate
            break
        if (candidate / "openai-agents-python-preview" / "src" / "agents").exists():
            repo_root = candidate / "openai-agents-python-preview"
            break

src_root = repo_root / "src"
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

env_path = None
for candidate in [Path.cwd(), *Path.cwd().parents, repo_root, *repo_root.parents]:
    maybe_env = candidate / ".env"
    if maybe_env.exists():
        env_path = maybe_env
        break

if env_path is not None:
    try:
        from dotenv import load_dotenv

        load_dotenv(env_path, override=False)
    except Exception:
        for raw_line in env_path.read_text().splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

required_modules = ["agents", "openai", "e2b", "e2b_code_interpreter"]
missing_modules = []
for module_name in required_modules:
    try:
        importlib.import_module(module_name)
    except Exception:
        missing_modules.append(module_name)

if missing_modules:
    print(f"Installing missing modules into kernel env {sys.executable}: {missing_modules}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", f"{repo_root}[e2b]"])
    importlib.invalidate_caches()
    print("Install complete. If imports still fail, restart the kernel and rerun from the top.")
else:
    print(f"Kernel env already has required modules: {required_modules}")

print(f"Using repo root: {repo_root}")
print(f"OPENAI_API_KEY set: {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"E2B_API_KEY set: {bool(os.getenv('E2B_API_KEY'))}")

In [ ]:
from pydantic import BaseModel, Field

from agents import Agent, ModelSettings, Runner
from agents.extensions.sandbox import (
    E2BSandboxClient,
    E2BSandboxClientOptions,
    E2BSandboxType,
)
from agents.run import RunConfig
from agents.sandbox import SandboxAgent, SandboxRunConfig
from examples.sandbox.misc.example_support import text_manifest
from examples.sandbox.misc.workspace_shell import WorkspaceShellCapability

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4")
SANDBOX_TYPE = E2BSandboxType(os.getenv("E2B_SANDBOX_TYPE", E2BSandboxType.E2B.value))
TEMPLATE = os.getenv("E2B_TEMPLATE") or None
TIMEOUT = int(os.getenv("E2B_TIMEOUT", "300"))


def make_e2b_run_config(*, exposed_ports: tuple[int, ...] = ()) -> RunConfig:
    return RunConfig(
        sandbox=SandboxRunConfig(
            client=E2BSandboxClient(),
            options=E2BSandboxClientOptions(
                sandbox_type=SANDBOX_TYPE,
                template=TEMPLATE,
                timeout=TIMEOUT,
                exposed_ports=exposed_ports,
            ),
        )
    )


class TestLaneResult(BaseModel):
    status: str = Field(description="pass or fail")
    summary: str
    evidence_files: list[str] = Field(min_length=1)


class LintLaneResult(BaseModel):
    status: str = Field(description="pass or fail")
    summary: str
    evidence_files: list[str] = Field(min_length=1)


class PreviewLaneResult(BaseModel):
    status: str = Field(description="pass or fail")
    summary: str
    app_port: int
    evidence_files: list[str] = Field(min_length=1)

In [ ]:
test_manifest = text_manifest(
    {
        "README.md": "Tiny app repo for CI sandbox example.\n",
        "tests_report.txt": "3 tests collected\n3 passed\n",
        "run_tests.sh": "#!/bin/sh\ncat tests_report.txt > test_results.txt\nexit 0\n",
    }
)

lint_manifest = text_manifest(
    {
        "README.md": "Tiny app repo for CI sandbox example.\n",
        "lint_report.txt": "ruff: clean\nformatting: clean\n",
        "run_lint.sh": "#!/bin/sh\ncat lint_report.txt > lint_results.txt\nexit 0\n",
    }
)

preview_manifest = text_manifest(
    {
        "README.md": "Tiny app repo for CI sandbox example.\n",
        "index.html": (
            "<html><body><h1>Sandbox Preview</h1><p>CI preview lane is live.</p></body></html>"
        ),
        "preview_notes.txt": "Preview server should listen on port 8765.\n",
    }
)

test_runner = SandboxAgent(
    name="Test Runner",
    model=MODEL,
    instructions=(
        "You are a CI test lane. Inspect the workspace, run the test script, and return a "
        "structured summary."
    ),
    developer_instructions=(
        "Use the shell tool. Run ./run_tests.sh and summarize the result using only files you "
        "actually inspected or created."
    ),
    default_manifest=test_manifest,
    capabilities=[WorkspaceShellCapability()],
    model_settings=ModelSettings(tool_choice="required"),
    output_type=TestLaneResult,
)

lint_runner = SandboxAgent(
    name="Lint Runner",
    model=MODEL,
    instructions=(
        "You are a CI lint lane. Inspect the workspace, run the lint script, and return a "
        "structured summary."
    ),
    developer_instructions=(
        "Use the shell tool. Run ./run_lint.sh and summarize the result using only files you "
        "actually inspected or created."
    ),
    default_manifest=lint_manifest,
    capabilities=[WorkspaceShellCapability()],
    model_settings=ModelSettings(tool_choice="required"),
    output_type=LintLaneResult,
)

preview_runner = SandboxAgent(
    name="Preview Runner",
    model=MODEL,
    instructions=(
        "You are a CI preview lane. Inspect the workspace, start a tiny HTTP server for the app, "
        "and return a structured summary."
    ),
    developer_instructions=(
        "Use the shell tool. Start a background server with `python -m http.server 8765`. "
        "Do not invent ports. app_port must be 8765."
    ),
    default_manifest=preview_manifest,
    capabilities=[WorkspaceShellCapability()],
    model_settings=ModelSettings(tool_choice="required"),
    output_type=PreviewLaneResult,
)

In [ ]:
preview_client = E2BSandboxClient()
preview_session = await preview_client.create(
    manifest=preview_manifest,
    options=E2BSandboxClientOptions(
        sandbox_type=SANDBOX_TYPE,
        template=TEMPLATE,
        timeout=TIMEOUT,
        exposed_ports=(8765,),
    ),
)
await preview_session.start()


async def structured_output_with_preview(result) -> str:
    final_output = result.final_output
    payload = (
        final_output.model_dump(mode="json")
        if isinstance(final_output, BaseModel)
        else {"raw": str(final_output)}
    )
    if payload.get("app_port"):
        endpoint = await preview_session.resolve_exposed_port(int(payload["app_port"]))
        payload["preview_url"] = endpoint.url_for("http")
    return json.dumps(payload, sort_keys=True)


orchestrator = Agent(
    name="CI Sandbox Coordinator",
    model=MODEL,
    instructions=(
        "You coordinate CI lanes. You must call `run_tests_lane`, `run_lint_lane`, and "
        "`run_preview_lane`, then synthesize a concise CI status report with pass/fail, key "
        "evidence files, and the preview URL when available. The preview lane tool output "
        "includes a `preview_url` field that was resolved from the sandbox session. Use that "
        "exact `preview_url` value in the final answer and never replace it with localhost or "
        "an invented URL."
    ),
    model_settings=ModelSettings(tool_choice="required"),
    tools=[
        test_runner.as_tool(
            tool_name="run_tests_lane",
            tool_description="Run the test lane in its own isolated sandbox.",
            custom_output_extractor=structured_output_with_preview,
            run_config=make_e2b_run_config(),
        ),
        lint_runner.as_tool(
            tool_name="run_lint_lane",
            tool_description="Run the lint lane in its own isolated sandbox.",
            custom_output_extractor=structured_output_with_preview,
            run_config=make_e2b_run_config(),
        ),
        preview_runner.as_tool(
            tool_name="run_preview_lane",
            tool_description="Run the preview lane in its own isolated sandbox.",
            custom_output_extractor=structured_output_with_preview,
            run_config=RunConfig(sandbox=SandboxRunConfig(session=preview_session)),
        ),
    ],
)

question = (
    "Run the three CI lanes for this tiny repo. Return a short status summary with one line per "
    "lane, mention the evidence files, and include the preview URL."
)

result = await Runner.run(orchestrator, question)  # type: ignore[top-level-await]  # noqa: F704
print(result.final_output)